# 🏦 Credit Vista Dataset — Data Cleaning, Feature Engineering & EDA
**Dataset:** `credit_vista_dataset.csv`  
**Records:** 50,000 | **Columns:** 42  
**Tasks:** Credit Score Regression + Creditworthiness Classification  

---


In [ ]:
# ── 1. IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

print("✅ All libraries imported successfully.")


In [ ]:
# ── 2. LOAD DATA ─────────────────────────────────────────────────────────────
df = pd.read_csv("credit_vista_dataset.csv")

print(f"Shape       : {df.shape}")
print(f"Rows        : {df.shape[0]:,}")
print(f"Columns     : {df.shape[1]}")
print()
df.head()


In [ ]:
# ── 3. DATA TYPES & BASIC INFO ───────────────────────────────────────────────
df.info()


In [ ]:
# ── 4. MISSING VALUES CHECK ──────────────────────────────────────────────────
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "✅ No missing values found in the dataset!")


In [ ]:
# ── 5. DUPLICATE ROWS CHECK ──────────────────────────────────────────────────
dup_count = df.duplicated().sum()
print(f"Duplicate rows: {dup_count}")
uid_unique = df['user_id'].nunique()
print(f"Unique user_ids: {uid_unique} / {len(df)} → {'✅ All unique' if uid_unique == len(df) else '⚠ Duplicates in user_id!'}")


In [ ]:
# ── 6. DESCRIPTIVE STATISTICS ────────────────────────────────────────────────
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean','std'])


## 📊 7. Target Variable Analysis

In [ ]:
# ── 7a. Credit Score Distribution (Regression Target) ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['credit_score'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['credit_score'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean = {df['credit_score'].mean():.0f}")
axes[0].axvline(df['credit_score'].median(), color='orange', linestyle='--', linewidth=1.5, label=f"Median = {df['credit_score'].median():.0f}")
axes[0].set_title("Credit Score Distribution")
axes[0].set_xlabel("Credit Score")
axes[0].set_ylabel("Count")
axes[0].legend()

# Risk tier breakdown
risk_order = ['Poor', 'Below Average', 'Fair', 'Good', 'Excellent']
risk_counts = df['risk_tier'].value_counts().reindex(risk_order)
colors = ['#d73027','#fc8d59','#fee08b','#91cf60','#1a9850']
axes[1].bar(risk_counts.index, risk_counts.values, color=colors, edgecolor='white')
axes[1].set_title("Risk Tier Distribution")
axes[1].set_xlabel("Risk Tier")
axes[1].set_ylabel("Count")
for i, v in enumerate(risk_counts.values):
    axes[1].text(i, v + 200, f"{v:,}", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig("01_target_distribution.png", dpi=120)
plt.show()
print(f"Credit Score — Mean: {df['credit_score'].mean():.1f} | Std: {df['credit_score'].std():.1f} | Min: {df['credit_score'].min()} | Max: {df['credit_score'].max()}")


In [ ]:
# ── 7b. Creditworthy Class Balance (Classification Target) ───────────────────
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
counts = df['creditworthy'].value_counts()
labels = ['Not Creditworthy (0)', 'Creditworthy (1)']
ax.pie(counts.values, labels=labels, autopct='%1.1f%%',
       colors=['#e05c5c','#5cb85c'], startangle=90,
       wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title("Creditworthy Class Distribution")
plt.tight_layout()
plt.savefig("02_class_balance.png", dpi=120)
plt.show()
print(counts)


## 📋 8. Categorical Feature Analysis

In [ ]:
# ── 8. Categorical Features vs Credit Score ──────────────────────────────────
cat_cols = ['gender_label', 'employment_type_label', 'city_tier_label', 'education_label']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = df.groupby(col)['credit_score'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='credit_score', order=order, ax=axes[i],
                palette='Set2', width=0.5)
    axes[i].set_title(f"Credit Score by {col.replace('_label','').replace('_',' ').title()}")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Credit Score")
    axes[i].tick_params(axis='x', rotation=20)

plt.suptitle("Categorical Features vs Credit Score", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("03_categorical_vs_credit_score.png", dpi=120)
plt.show()


## 📈 9. Numeric Feature Distributions

In [ ]:
# ── 9. Numeric Feature Distributions ────────────────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['credit_score','creditworthy','gender','employment_type','city_tier','education']]

fig, axes = plt.subplots(6, 5, figsize=(20, 20))
axes = axes.flatten()

for i, col in enumerate(numeric_cols[:30]):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col.replace('_',' '), fontsize=9)
    axes[i].tick_params(labelsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Numeric Feature Distributions", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("04_numeric_distributions.png", dpi=120)
plt.show()


## 🔥 10. Correlation Analysis

In [ ]:
# ── 10. Correlation Heatmap ──────────────────────────────────────────────────
# Drop redundant label columns and ID for correlation
drop_for_corr = ['user_id','gender_label','employment_type_label','city_tier_label','education_label','risk_tier']
corr_df = df.drop(columns=drop_for_corr)

corr_matrix = corr_df.corr()

plt.figure(figsize=(18, 15))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, annot=False, linewidths=0.3,
            cbar_kws={'shrink': 0.6})
plt.title("Correlation Heatmap — All Numeric Features", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("05_correlation_heatmap.png", dpi=120)
plt.show()


In [ ]:
# ── 10b. Top Correlations with Credit Score ──────────────────────────────────
credit_corr = corr_matrix['credit_score'].drop('credit_score').sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
colors = ['#d73027' if v < 0 else '#1a9850' for v in credit_corr.values]
bars = ax.barh(credit_corr.index, credit_corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title("Feature Correlation with Credit Score", fontsize=13, fontweight='bold')
ax.set_xlabel("Pearson Correlation Coefficient")

for bar, val in zip(bars, credit_corr.values):
    ax.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.savefig("06_correlations_with_credit_score.png", dpi=120)
plt.show()


## 🔍 11. Outlier Detection

In [ ]:
# ── 11. Outlier Detection using IQR ──────────────────────────────────────────
key_cols = ['declared_monthly_income','monthly_expenses','avg_upi_txn_amount',
            'job_tenure_years','emi_burden_ratio','savings_ratio','upi_txn_count_monthly']

outlier_summary = []
for col in key_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append({'Feature': col, 'Q1': round(Q1,2), 'Q3': round(Q3,2),
                             'IQR': round(IQR,2), 'Lower_Fence': round(lower,2),
                             'Upper_Fence': round(upper,2), 'Outlier_Count': n_out,
                             'Outlier_%': round(n_out/len(df)*100, 2)})

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))

# Boxplots for top features
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(key_cols):
    axes[i].boxplot(df[col], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i].set_title(col.replace('_',' '), fontsize=8)
    axes[i].tick_params(labelsize=7)
axes[-1].set_visible(False)
plt.suptitle("Boxplots — Key Features (Outlier View)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("07_boxplots_outliers.png", dpi=120)
plt.show()


## 🔗 12. Bivariate Analysis — Key Features vs Credit Score

In [ ]:
# ── 12. Scatter plots of top correlated features vs Credit Score ─────────────
top_features = ['financial_discipline_index','payment_consistency',
                'income_regularity','savings_ratio','emi_burden_ratio','default_history']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    c = df['creditworthy'].map({0:'#e05c5c', 1:'#5cb85c'})
    axes[i].scatter(df[feat], df['credit_score'], c=c, alpha=0.15, s=5)
    axes[i].set_xlabel(feat.replace('_',' '))
    axes[i].set_ylabel("Credit Score")
    axes[i].set_title(f"{feat.replace('_',' ')} vs Credit Score")

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#5cb85c', label='Creditworthy'),
                   Patch(facecolor='#e05c5c', label='Not Creditworthy')]
fig.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.suptitle("Bivariate Analysis — Key Features vs Credit Score", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("08_bivariate_analysis.png", dpi=120)
plt.show()


## 🧹 13. Data Cleaning

In [ ]:
# ── 13a. Drop Redundant / Leakage Columns ────────────────────────────────────
# user_id       → identifier, not a feature
# *_label       → string duplicates of encoded numerics
# annual_income → = declared_monthly_income × 12 (multicollinearity)
# avg_monthly_credit → ~0.98 correlation with declared_monthly_income
# risk_tier     → derived directly from credit_score (leakage for regression)

drop_cols = [
    'user_id',
    'gender_label',
    'employment_type_label',
    'city_tier_label',
    'education_label',
    'annual_income',        # redundant with declared_monthly_income
    'avg_monthly_credit',   # near-duplicate of declared_monthly_income
    'risk_tier',            # label derived from target (credit_score)
]

df_clean = df.drop(columns=drop_cols)
print(f"Original shape : {df.shape}")
print(f"Cleaned shape  : {df_clean.shape}")
print(f"Columns dropped: {drop_cols}")


In [ ]:
# ── 13b. Verify Remaining Columns ────────────────────────────────────────────
print("Remaining columns:")
for i, col in enumerate(df_clean.columns, 1):
    print(f"  {i:2d}. {col}")
print(f"\nMissing values after cleaning: {df_clean.isnull().sum().sum()}")


## ⚙️ 14. Feature Engineering

In [ ]:
# ── 14a. New Derived Features ────────────────────────────────────────────────

# 1. Income-to-expense ratio: financial buffer
df_clean['income_expense_ratio'] = df_clean['declared_monthly_income'] / (df_clean['monthly_expenses'] + 1)

# 2. Net savings monthly: actual money saved
df_clean['net_savings_monthly'] = df_clean['declared_monthly_income'] * df_clean['savings_ratio']

# 3. Loan stress score: combines EMI burden × number of loans × default history
df_clean['loan_stress_score'] = (df_clean['emi_burden_ratio'] * df_clean['num_active_loans'] +
                                  df_clean['default_history'] * 0.5)

# 4. Payment reliability score: average of payment-related scores
df_clean['payment_reliability'] = (df_clean['payment_consistency'] +
                                    df_clean['utility_payment_regularity'] +
                                    df_clean['rent_payment_on_time'] +
                                    df_clean['mobile_recharge_consistency']) / 4

# 5. Stability index: employment stability proxy
df_clean['stability_index'] = (df_clean['job_tenure_years'] / (df_clean['job_changes_5yr'] + 1)) * df_clean['has_social_security']

# 6. Digital trust score: digital payment engagement vs cash dependency
df_clean['digital_trust_score'] = df_clean['digital_engagement_score'] * (1 - df_clean['cash_dependency'])

# 7. Age-income interaction: proxy for career stage × income
df_clean['age_income_interaction'] = df_clean['age'] * df_clean['declared_monthly_income'] / 1e6

print("✅ New features engineered:")
new_feats = ['income_expense_ratio','net_savings_monthly','loan_stress_score',
             'payment_reliability','stability_index','digital_trust_score','age_income_interaction']
for f in new_feats:
    print(f"  • {f:30s} | mean={df_clean[f].mean():.4f} | std={df_clean[f].std():.4f}")

print(f"\nFinal dataset shape: {df_clean.shape}")


In [ ]:
# ── 14b. Check Correlation of New Features with Target ───────────────────────
new_feats_corr = df_clean[new_feats + ['credit_score']].corr()['credit_score'].drop('credit_score')
print("New Feature Correlations with Credit Score:")
print(new_feats_corr.sort_values(ascending=False).to_string())

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#1a9850' if v > 0 else '#d73027' for v in new_feats_corr.sort_values().values]
ax.barh(new_feats_corr.sort_values().index, new_feats_corr.sort_values().values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title("New Engineered Features — Correlation with Credit Score")
ax.set_xlabel("Pearson r")
plt.tight_layout()
plt.savefig("09_engineered_features_correlation.png", dpi=120)
plt.show()


## 📉 15. Advanced EDA — Group Analysis

In [ ]:
# ── 15a. Credit Score by Employment Type and Default History ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Employment type
emp_map = {0:'Business Owner', 1:'Govt Salaried', 2:'Gig/Freelance', 3:'Private Salaried', 4:'Self Employed'}
df_temp = df.copy()
df_temp['emp_label'] = df_temp['employment_type'].map(emp_map)
order = df_temp.groupby('emp_label')['credit_score'].median().sort_values(ascending=False).index
sns.violinplot(data=df_temp, x='emp_label', y='credit_score', order=order,
               ax=axes[0], palette='pastel', inner='box')
axes[0].set_title("Credit Score Distribution by Employment Type")
axes[0].set_xlabel("")
axes[0].tick_params(axis='x', rotation=20)

# Default history
sns.boxplot(data=df, x='default_history', y='credit_score', ax=axes[1],
            palette=['#5cb85c','#e05c5c'])
axes[1].set_xticklabels(['No Default (0)', 'Has Default (1)'])
axes[1].set_title("Credit Score by Default History")
axes[1].set_xlabel("")

plt.tight_layout()
plt.savefig("10_group_analysis.png", dpi=120)
plt.show()


In [ ]:
# ── 15b. Heatmap: Credit Score by Education × City Tier ──────────────────────
pivot = df.pivot_table(values='credit_score',
                       index='education_label',
                       columns='city_tier_label',
                       aggfunc='mean')

edu_order = ['Below_10th','Higher_Secondary','Graduate','Postgraduate']
city_order = ['Metro','Tier2','Tier3']
pivot = pivot.reindex(index=[e for e in edu_order if e in pivot.index],
                      columns=[c for c in city_order if c in pivot.columns])

plt.figure(figsize=(7, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Avg Credit Score'})
plt.title("Average Credit Score: Education × City Tier")
plt.xlabel("City Tier")
plt.ylabel("Education Level")
plt.tight_layout()
plt.savefig("11_education_city_heatmap.png", dpi=120)
plt.show()


In [ ]:
# ── 15c. Distribution of Financial Discipline Index by Creditworthy ──────────
fig, ax = plt.subplots(figsize=(8, 5))
for label, color in [(0,'#e05c5c'),(1,'#5cb85c')]:
    subset = df_clean[df_clean['creditworthy'] == label]['financial_discipline_index']
    ax.hist(subset, bins=50, alpha=0.6, color=color, label=f"{'Creditworthy' if label else 'Not Creditworthy'}", density=True)
ax.set_xlabel("Financial Discipline Index")
ax.set_ylabel("Density")
ax.set_title("Financial Discipline Index Distribution by Creditworthiness")
ax.legend()
plt.tight_layout()
plt.savefig("12_discipline_index_by_creditworthy.png", dpi=120)
plt.show()


## 💾 16. Save Cleaned & Feature-Engineered Dataset

In [ ]:
# ── 16. Save Final Cleaned Dataset ───────────────────────────────────────────
output_path = "credit_vista_cleaned.csv"
df_clean.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved to: {output_path}")
print(f"   Shape: {df_clean.shape}")
print(f"   Columns ({df_clean.shape[1]}):")
for c in df_clean.columns:
    print(f"     • {c}")


## ✅ 17. EDA Summary & Findings

### Key Findings:
| # | Finding |
|---|---------|
| 1 | **Zero missing values** — dataset is complete, no imputation needed |
| 2 | **Credit score** follows a near-normal distribution centered ~632 (range 300–900) |
| 3 | **Class balance** is mild (65% creditworthy / 35% not) — no SMOTE required |
| 4 | **Strongest positive correlators** with credit score: `financial_discipline_index`, `payment_consistency`, `income_regularity` |
| 5 | **Strongest negative correlators**: `default_history`, `emi_burden_ratio`, `cash_dependency` |
| 6 | **Govt Salaried** employees have the highest median credit scores; **Gig/Freelance** the lowest |
| 7 | **Higher education + Metro city** combination yields highest average credit scores |
| 8 | **Engineered features** like `loan_stress_score`, `payment_reliability`, `digital_trust_score` show meaningful correlations |
| 9 | `annual_income` and `avg_monthly_credit` were dropped due to near-perfect multicollinearity |
| 10 | `risk_tier` was dropped to prevent **data leakage** in the regression task |

### Ready for Modeling:
- Cleaned CSV saved as **`credit_vista_cleaned.csv`**
- 34 features remain after cleaning + 7 engineered features = **41 total features**
- Targets: `credit_score` (regression) and `creditworthy` (classification)
